<a target="_blank" href="https://colab.research.google.com/github/jangmino/sam2/blob/ewha-drape/notebooks/drape_preprocess.ipynb">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

# Enviroment Set-up

In [ ]:
using_colab = False

In [ ]:
if using_colab:
    import torch
    import torchvision
    print("PyTorch version:", torch.__version__)
    print("Torchvision version:", torchvision.__version__)
    print("CUDA is available:", torch.cuda.is_available())
    import sys
    !{sys.executable} -m pip install opencv-python matplotlib
    !{sys.executable} -m pip install 'git+https://github.com/facebookresearch/sam2.git'

    !mkdir -p videos
    !wget -P videos https://dl.fbaipublicfiles.com/segment_anything_2/assets/bedroom.zip
    !unzip -d videos videos/bedroom.zip

    !mkdir -p ../checkpoints/
    !wget -P ../checkpoints/ https://dl.fbaipublicfiles.com/segment_anything_2/092824/sam2.1_hiera_large.pt

## Set-up

In [ ]:
import os
import shutil
import uuid
import glob
import subprocess
from typing import Dict, List, Tuple, Any

import gradio as gr
from PIL import Image, ImageDraw
import tempfile
import debugpy

import numpy as np

import cv2
import json
import math

In [ ]:
# if using Apple MPS, fall back to CPU for unsupported ops
os.environ["PYTORCH_ENABLE_MPS_FALLBACK"] = "1"
import numpy as np
import torch
import matplotlib.pyplot as plt
from PIL import Image

In [ ]:
# select the device for computation
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")
print(f"using device: {device}")

if device.type == "cuda":
    # use bfloat16 for the entire notebook
    torch.autocast("cuda", dtype=torch.bfloat16).__enter__()
    # turn on tfloat32 for Ampere GPUs (https://pytorch.org/docs/stable/notes/cuda.html#tensorfloat-32-tf32-on-ampere-devices)
    if torch.cuda.get_device_properties(0).major >= 8:
        torch.backends.cuda.matmul.allow_tf32 = True
        torch.backends.cudnn.allow_tf32 = True
elif device.type == "mps":
    print(
        "\nSupport for MPS devices is preliminary. SAM 2 is trained with CUDA and might "
        "give numerically different outputs and sometimes degraded performance on MPS. "
        "See e.g. https://github.com/pytorch/pytorch/issues/84936 for a discussion."
    )

### Loading the SAM 2 video predictor

In [ ]:
from sam2.build_sam import build_sam2_video_predictor

sam2_checkpoint = "../checkpoints/sam2.1_hiera_large.pt"
# sam2_checkpoint = "../checkpoints/sam2.1_hiera_small.pt"
model_cfg = "configs/sam2.1/sam2.1_hiera_l.yaml"
# model_cfg = "configs/sam2.1/sam2.1_hiera_s.yaml"

predictor = build_sam2_video_predictor(model_cfg, sam2_checkpoint, device=device)

### Utilities

In [ ]:
# -----------------------------
# Utilities
# -----------------------------
def extract_frames(video_path: str, workdir: str) -> List[str]:
    """
    Use ffmpeg to extract frames to workdir/frames/%05d.jpg
    Returns list of frame paths sorted.
    """
    frames_dir = os.path.join(workdir, "frames")
    os.makedirs(frames_dir, exist_ok=True)

    # -vsync 0 keeps all frames; adjust as needed.
    # -qscale:v 2 yields decent JPEG quality; adjust if you want PNG.
    cmd = [
        "ffmpeg", "-y",
        "-i", video_path,
        "-qscale:v", "2",
        "-vsync", "0",
        os.path.join(frames_dir, "%05d.jpg"),
    ]
    subprocess.run(cmd, check=True, stdout=subprocess.PIPE, stderr=subprocess.PIPE)

    frame_paths = sorted(glob.glob(os.path.join(frames_dir, "*.jpg")))
    return frame_paths


def draw_points_on_frame(frame_path: str,
                         obj_points: Dict[str, Dict[str, List[Tuple[int, int, int]]]],
                         ref_points: Dict[str, List[Tuple[int, int, int]]],
                         frame_idx: int) -> Image.Image:
    """
    Overlay points for the given frame_idx.
    - obj_points[object_name]["add"|"remove"] -> list[(x,y,frame_idx)]
    - ref_points["Ref 1"/"Ref 2"] -> list[(x,y,frame_idx)]
    """
    im = Image.open(frame_path).convert("RGB")
    draw = ImageDraw.Draw(im)

    # Objects: add/remove
    for obj_name, buckets in obj_points.items():
        for mode, pts in buckets.items():
            for (x, y, f) in pts:
                if f != frame_idx:
                    continue
                # add 포인트는 원, remove 포인트는 X 마크로 표시
                r = 6
                if mode == "add":
                    draw.ellipse((x - r, y - r, x + r, y + r), outline=(0, 255, 0), width=2)
                else:
                    # draw an X
                    draw.line((x - r, y - r, x + r, y + r), fill=(255, 0, 0), width=2)
                    draw.line((x - r, y + r, x + r, y - r), fill=(255, 0, 0), width=2)

    # Reference points: small squares
    for ref_name, pts in ref_points.items():
        for (x, y, f) in pts:
            if f != frame_idx:
                continue
            r = 5
            draw.rectangle((x - r, y - r, x + r, y + r), outline=(0, 200, 255), width=2)

    return im



# Gradio App Codes

In [ ]:
# -----------------------------
# Gradio App (css styles)
# -----------------------------
CSS = '''
:root { --scale-font: 1; }
html, body, .gradio-container { font-size: calc(13px * var(--scale-font)); }

/* Scale fonts on larger screens (HD/FullHD) but keep modest */
@media (min-width: 1600px) and (min-height: 900px) { :root { --scale-font: 1.0; } }
@media (min-width: 1920px) and (min-height: 1080px) { :root { --scale-font: 1.05; } }

/* Left column: keep toolbar compact */
#left_col .gr-video, #left_col video {
  max-height: clamp(150px, 20vh, 240px);
  width: 100%;
  object-fit: contain;
}
#left_col {
  max-height: 90vh;
  overflow: auto;
  padding-right: 8px;
}

/* Right column: stable layout; progress at bottom */
#right_col {
  display: flex;
  flex-direction: column;
  height: 90vh;
  gap: 8px;
}
#right_col .gr-row, #right_col .gr-block { flex: 0 0 auto; }

/* Frame image constraints */
#frame_img img {
  max-width: 100%;
  max-height: 55vh;
  object-fit: contain;
  display: block;
  margin: 0 auto;
}
#frame_img {
  max-height: 55vh;
  overflow: auto;
}

/* Controls readability but slightly smaller */
.gr-button, .gr-radio, .gr-dropdown, .gr-textbox, .gr-slider, .gr-markdown {
  font-size: 0.95rem;
}
h1, h2, h3 { line-height: 1.2; }
.gr-slider input[type="range"] { height: 6px; }

/* Make progress_log take less vertical space */
#right_col .gr-textbox { margin-top: auto; }
#right_col .gr-textbox textarea {
  min-height: 2.8em;
  max-height: 4em;
  line-height: 1.2;
  font-size: 0.95rem;
  resize: vertical;
}
'''

css_inject = """
<style>
:root { --scale-font: 1; }
html, body, .gradio-container { font-size: calc(13px * var(--scale-font)) !important; }
@media (min-width:1600px) and (min-height:900px) { :root { --scale-font: 1.0; } }
@media (min-width:1920px) and (min-height:1080px) { :root { --scale-font: 1.05; } }

/* Shared sizing for image/canvas */
#frame_img, .gr-img, .gr-image, .gradio-image { max-height: 55vh !important; overflow: auto !important; }
#frame_img img, #frame_img canvas, .gr-image img, .gr-image canvas, .gradio-image img, .gradio-image canvas {
  max-height: 55vh !important; max-width: 100% !important; height: auto !important; object-fit: contain !important; display: block !important; margin: 0 auto !important;
}

/* Keep right column stable and progress log at bottom */
#right_col { display: flex !important; flex-direction: column !important; height: 90vh !important; }
#right_col > .gr-row, #right_col > .gr-block { flex: 0 0 auto !important; }
#right_col .gr-textbox { margin-top: auto !important; }
#right_col .gr-textbox textarea { min-height: 2.8em !important; max-height: 4em !important; line-height: 1.2 !important; font-size: 0.95rem !important; resize: vertical !important; }

/* Control readability */
.gr-button, .gr-radio, .gr-dropdown, .gr-textbox, .gr-slider, .gr-markdown { font-size: 0.95rem !important; }
.gr-slider input[type="range"] { height: 6px !important; }
</style>
"""

In [ ]:
class DrapeApp:
    """Encapsulated Gradio app for drape preprocessing.

    This class holds all callbacks and helpers. build_ui() constructs the UI and wires events.
    """
    def __init__(self, predictor, css: str, css_inject: str):
        self.predictor = predictor
        self.css = css
        self.css_inject = css_inject

    # -----------------------------
    # Helper utilities (class-scoped)
    # -----------------------------
    def _get_or_assign_color(self, state: Dict[str, Any], obj_name: str):
        if obj_name in state["obj_colors"]:
            return state["obj_colors"][obj_name]
        cmap = plt.get_cmap("tab10")
        idx = (state.get("next_obj_id", 1) - 1) % 10
        rgba = tuple(int(c * 255) for c in cmap(idx)[:3]) + (150,)
        state["obj_colors"][obj_name] = rgba
        return rgba

    def _overlay_masks_on_image(self, state: Dict[str, Any], frame_path: str, masks_dict: Dict[int, np.ndarray], id_to_name: Dict[int, str], frame_idx: int = 0):
        im = Image.open(frame_path).convert("RGBA")
        overlay = Image.new("RGBA", im.size, (0, 0, 0, 0))
        for obj_id, mask in masks_dict.items():
            if mask is None:
                continue
            obj_name = id_to_name.get(obj_id, f"Obj{obj_id}")
            color = self._get_or_assign_color(state, obj_name)
            color_img = Image.new("RGBA", im.size, color)
            try:
                mask_img = Image.fromarray((mask.astype("uint8") * 255).astype("uint8"))
            except Exception:
                mask_img = Image.fromarray(mask.astype("uint8"))
            mask_img = mask_img.convert("L")
            overlay.paste(color_img, (0, 0), mask_img)
        composed = Image.alpha_composite(im, overlay).convert("RGB")

        if frame_idx == 0:
            draw = ImageDraw.Draw(composed)
            objs = state.get("objects", {})
            for oid, name in id_to_name.items():
                buckets = objs.get(name, {"add": [], "remove": []})
                for (x, y, f) in buckets.get("add", []):
                    if f != 0:
                        continue
                    r = 6
                    draw.ellipse((int(x) - r, int(y) - r, int(x) + r, int(y) + r), outline=(0, 255, 0), width=2)
                for (x, y, f) in buckets.get("remove", []):
                    if f != 0:
                        continue
                    r = 6
                    draw.line((int(x) - r, int(y) - r, int(x) + r, int(y) + r), fill=(255, 0, 0), width=2)
                    draw.line((int(x) - r, int(y) + r, int(x) + r, int(y) - r), fill=(255, 0, 0), width=2)

        out_path = os.path.join(state["workdir"], f"overlay_masks_{uuid.uuid4().hex[:8]}.jpg")
        composed.save(out_path)
        return out_path

    def _normalize_mask_array(self, mask) -> np.ndarray:
        try:
            import torch as _torch
            if _torch.is_tensor(mask):
                arr = mask.detach().cpu().numpy()
            else:
                arr = np.asarray(mask)
        except Exception:
            arr = np.asarray(mask)
        arr = np.squeeze(arr)
        if arr.ndim > 2:
            try:
                arr = arr.reshape(arr.shape[-2], arr.shape[-1])
            except Exception:
                raise ValueError(f"Cannot normalize mask with shape {arr.shape}")
        if arr.ndim != 2:
            raise ValueError(f"Mask has wrong number of dims after normalization: {arr.ndim}")
        try:
            mask_bool = (arr > 0).astype('uint8')
        except Exception:
            mask_bool = (np.asarray(arr).astype('float32') > 0).astype('uint8')
        return mask_bool

    def _pil_from_path(self, path: str):
        if path is None:
            return None
        try:
            im = Image.open(path).convert("RGB")
            return im
        except Exception as e:
            print(f"[_pil_from_path] failed to open {path}: {e}")
            return None

    # -----------------------------
    # Callbacks
    # -----------------------------
    def on_process(self, video_data, state: Dict[str, Any]):
        if state.get("busy"):
            raise gr.Error("Busy processing. Wait until current operation finishes.")
        if video_data is None:
            raise gr.Error("Please upload a video first.")
        base_tmp = tempfile.gettempdir()
        workdir = os.path.join(base_tmp, f"gr_sam2_{uuid.uuid4().hex}")
        os.makedirs(workdir, exist_ok=True)
        if isinstance(video_data, dict) and "name" in video_data:
            video_path = video_data["name"]
        else:
            video_path = str(video_data)
        try:
            frame_paths = extract_frames(video_path, workdir)
        except subprocess.CalledProcessError:
            shutil.rmtree(workdir, ignore_errors=True)
            raise gr.Error("ffmpeg failed to extract frames. Ensure ffmpeg is installed and the video is valid.")
        if not frame_paths:
            shutil.rmtree(workdir, ignore_errors=True)
            raise gr.Error("No frames were extracted from the video.")
        state.update({
            "workdir": workdir,
            "frame_paths": frame_paths,
            "objects": {},
            "refs": {"Ref 1": [], "Ref 2": []},
            "obj_name_to_id": {},
            "next_obj_id": 1,
            "obj_colors": {},
            "masks_per_frame": [{} for _ in frame_paths],
            "overlay_paths": [None for _ in frame_paths],
            "propagated": False,
            "busy": False,
        })
        frames_dir = os.path.join(workdir, "frames")
        try:
            inference_state = self.predictor.init_state(video_path=frames_dir)
            state["inference_state"] = inference_state
        except Exception as e:
            shutil.rmtree(workdir, ignore_errors=True)
            raise gr.Error(f"Failed to initialize predictor: {e}")
        first_frame = frame_paths[0]
        max_idx = len(frame_paths) - 1
        return (
            state,
            gr.update(value=self._pil_from_path(first_frame)),
            gr.update(minimum=0, maximum=max_idx, value=0, interactive=True),
            gr.update(choices=[], value=None),
            "Ready. Frames extracted and predictor initialized."
        )

    def on_add_object(self, state: Dict[str, Any]):
        if state.get("busy"):
            return state, gr.update(choices=list(state.get("objects", {}).keys()), value=None)
        objs = state["objects"]
        idx = 1
        while True:
            name = f"Object {idx}"
            if name not in objs:
                break
            idx += 1
        objs[name] = {"add": [], "remove": []}
        state["objects"] = objs
        obj_id = state.get("next_obj_id", 1)
        state.setdefault("obj_name_to_id", {})[name] = obj_id
        state["next_obj_id"] = obj_id + 1
        _ = self._get_or_assign_color(state, name)
        return state, gr.update(choices=list(objs.keys()), value=name)

    def on_remove_active_object(self, state: Dict[str, Any], active_obj: str, frame_idx: int):
        if state.get("busy"):
            return state, gr.update(choices=list(state.get("objects", {}).keys()), value=active_obj), gr.update(value=None), "Busy: cannot remove object now.", gr.update(value="")
        objs = state["objects"]
        if active_obj in objs:
            obj_map = state.get("obj_name_to_id", {})
            obj_id = obj_map.pop(active_obj, None)
            try:
                if obj_id is not None and state.get("inference_state") is not None:
                    self.predictor.remove_object(state["inference_state"], obj_id)
            except Exception:
                pass
            del objs[active_obj]
        state["objects"] = objs
        new_choices = list(objs.keys())
        new_value = new_choices[0] if new_choices else None
        if not state.get("frame_paths"):
            return state, gr.update(choices=new_choices, value=new_value), gr.update(value=None), "Removed object.", gr.update(value="")
        fidx = int(frame_idx)
        fidx = max(0, min(fidx, len(state["frame_paths"]) - 1))
        frame_path = state["frame_paths"][fidx]
        im = draw_points_on_frame(frame_path, state["objects"], state["refs"], fidx)
        out_path = os.path.join(state["workdir"], f"overlay_{fidx:05d}.jpg")
        im.save(out_path)
        return state, gr.update(choices=new_choices, value=new_value), gr.update(value=self._pil_from_path(out_path)), f"Removed {active_obj}.", gr.update(value="")

    def on_clear_object_points(self, state: Dict[str, Any], active_obj: str, frame_idx: int):
        if state.get("busy"):
            return state, "Busy: cannot clear points now.", gr.update(value=None)
        if active_obj and active_obj in state["objects"]:
            state["objects"][active_obj] = {"add": [], "remove": []}
        if not state.get("frame_paths"):
            return state, "Cleared points for selected object.", gr.update(value=None)
        fidx = int(frame_idx)
        fidx = max(0, min(fidx, len(state["frame_paths"]) - 1))
        frame_path = state["frame_paths"][fidx]
        im = draw_points_on_frame(frame_path, state["objects"], state["refs"], fidx)
        out_path = os.path.join(state["workdir"], f"overlay_{fidx:05d}.jpg")
        im.save(out_path)
        return state, "Cleared points for selected object.", gr.update(value=self._pil_from_path(out_path))

    def on_clear_ref_points(self, state: Dict[str, Any], active_ref_name: str, frame_idx: int):
        if state.get("busy"):
            return state, f"Busy: cannot clear ref points now.", gr.update(value=None)
        if active_ref_name in state["refs"]:
            state["refs"][active_ref_name] = []
        if not state.get("frame_paths"):
            return state, f"Cleared points for {active_ref_name}.", gr.update(value=None)
        fidx = int(frame_idx)
        fidx = max(0, min(fidx, len(state["frame_paths"]) - 1))
        frame_path = state["frame_paths"][fidx]
        im = draw_points_on_frame(frame_path, state["objects"], state["refs"], fidx)
        out_path = os.path.join(state["workdir"], f"overlay_{fidx:05d}.jpg")
        im.save(out_path)
        return state, f"Cleared points for {active_ref_name}.", gr.update(value=self._pil_from_path(out_path))

    def on_frame_change(self, state: Dict[str, Any], frame_idx: int):
        if not state["frame_paths"]:
            return gr.update(value=None)
        frame_idx = int(frame_idx)
        frame_idx = max(0, min(frame_idx, len(state["frame_paths"]) - 1))
        state["last_frame_idx"] = frame_idx
        overlay_paths = state.get("overlay_paths", [])
        if overlay_paths and overlay_paths[frame_idx]:
            return state, gr.update(value=self._pil_from_path(overlay_paths[frame_idx]))
        frame_path = state["frame_paths"][frame_idx]
        im = draw_points_on_frame(frame_path, state["objects"], state["refs"], frame_idx)
        out_path = os.path.join(state["workdir"], f"overlay_{frame_idx:05d}.jpg")
        im.save(out_path)
        return state, gr.update(value=self._pil_from_path(out_path))

    def on_click_frame(self, evt: gr.SelectData, state: Dict[str, Any], active_obj: str, point_mode_choice: str, active_ref_name: str, frame_idx: int):
        if state.get("busy"):
            return state, gr.update(), "Busy processing. Please wait until tracking completes."
        if not state["frame_paths"]:
            return state, gr.update(), "No video processed yet."
        x, y = evt.index
        fidx = int(frame_idx)
        updated_text = ""
        out_mask_overlay_path = None
        if active_obj:
            bucket = "add" if point_mode_choice == "Add point" else "remove"
            state["objects"].setdefault(active_obj, {"add": [], "remove": []})
            state["objects"][active_obj][bucket].append((x, y, fidx))
            updated_text = f"[{active_obj}] {bucket} point: (x={x}, y={y}, f={fidx})"
            try:
                obj_map = state.get("obj_name_to_id", {})
                obj_id = obj_map.get(active_obj)
                if obj_id is None:
                    obj_id = state.get("next_obj_id", 1)
                    obj_map[active_obj] = obj_id
                    state["next_obj_id"] = obj_id + 1
                    state["obj_name_to_id"] = obj_map
                    self._get_or_assign_color(state, active_obj)
                lbl = 1 if bucket == "add" else 0
                pts = np.array([[x, y]], dtype=np.float32)
                labels = np.array([lbl], dtype=np.int32)
                if state.get("inference_state") is not None:
                    _, out_obj_ids, out_mask_logits = self.predictor.add_new_points_or_box(
                        inference_state=state["inference_state"],
                        frame_idx=fidx,
                        obj_id=obj_id,
                        points=pts,
                        labels=labels,
                    )
                    masks_dict = {}
                    if out_mask_logits is not None:
                        try:
                            masks_np = out_mask_logits.cpu().numpy()
                        except Exception:
                            masks_np = out_mask_logits.numpy()
                        for i, oid in enumerate(out_obj_ids):
                            try:
                                mask_raw = masks_np[i]
                                mask_norm = self._normalize_mask_array(mask_raw)
                                masks_dict[oid] = mask_norm
                            except Exception as e:
                                print(f"[on_click_frame] unexpected mask shape for oid={oid}: {e}")
                                continue
                    id_to_name = {v: k for k, v in state.get("obj_name_to_id", {}).items()}
                    frame_path = state["frame_paths"][fidx]
                    if masks_dict:
                        out_mask_overlay_path = self._overlay_masks_on_image(state, frame_path, masks_dict, id_to_name, frame_idx=fidx)
                        mpf = state.get("masks_per_frame", [{} for _ in state.get("frame_paths", [])])
                        for oid, m in masks_dict.items():
                            mpf[fidx][oid] = m
                        state["masks_per_frame"] = mpf
            except Exception as e:
                updated_text += f" (predictor error: {e})"
        else:
            state["refs"].setdefault(active_ref_name, [])
            state["refs"][active_ref_name].append((x, y, fidx))
            updated_text = f"[{active_ref_name}] point: (x={x}, y={y}, f={fidx})"
        if out_mask_overlay_path is None:
            frame_path = state["frame_paths"][fidx]
            im = draw_points_on_frame(frame_path, state["objects"], state["refs"], fidx)
            out_path = os.path.join(state["workdir"], f"overlay_{fidx:05d}.jpg")
            im.save(out_path)
            image_update = gr.update(value=self._pil_from_path(out_path))
        else:
            image_update = gr.update(value=self._pil_from_path(out_mask_overlay_path))
        info_html = f"<code>{updated_text}</code>"
        return state, image_update, info_html

    def track_objects(self, state: Dict[str, Any]):
        if not state.get("frame_paths"):
            yield "No frames to process.", gr.update(value=None), gr.update(value=0)
            return
        if state.get("busy"):
            yield "Already running.", gr.update(value=None), gr.update(value=0)
            return
        n = len(state["frame_paths"])
        last_idx = int(state.get("last_frame_idx", 0)) if state.get("frame_paths") else 0
        state["busy"] = True
        state.setdefault("overlay_paths", [None for _ in range(n)])
        state.setdefault("masks_per_frame", [{} for _ in range(n)])
        yield "Propagation started...", gr.update(value=self._pil_from_path(state["frame_paths"][last_idx])), gr.update(value=last_idx)
        video_segments = {}
        id_to_name = {v: k for k, v in state.get("obj_name_to_id", {}).items()}
        try:
            for out_frame_idx, out_obj_ids, out_mask_logits in self.predictor.propagate_in_video(state["inference_state"]):
                frame_dict = {}
                for i, out_obj_id in enumerate(out_obj_ids):
                    try:
                        raw = out_mask_logits[i]
                        try:
                            import torch as _torch
                            if _torch.is_tensor(raw):
                                raw_np = raw.cpu().numpy()
                            else:
                                raw_np = np.asarray(raw)
                        except Exception:
                            raw_np = np.asarray(raw)
                        norm = self._normalize_mask_array(raw_np)
                        frame_dict[int(out_obj_id)] = norm
                    except Exception as e:
                        print(f"[track_objects] failed to convert mask for frame {out_frame_idx}, obj {out_obj_id}: {e}")
                out_frame_idx = int(out_frame_idx)
                video_segments[out_frame_idx] = frame_dict
                if 0 <= out_frame_idx < n:
                    state["masks_per_frame"][out_frame_idx] = frame_dict
                frame_path = state["frame_paths"][out_frame_idx]
                if frame_dict:
                    try:
                        overlay_path = self._overlay_masks_on_image(state, frame_path, frame_dict, id_to_name, frame_idx=out_frame_idx)
                    except Exception as e:
                        print(f"[track_objects] overlay creation failed for frame {out_frame_idx}: {e}")
                        im = draw_points_on_frame(frame_path, state["objects"], state["refs"], out_frame_idx)
                        overlay_path = os.path.join(state["workdir"], f"overlay_{out_frame_idx:05d}.jpg")
                        im.save(overlay_path)
                else:
                    im = draw_points_on_frame(frame_path, state["objects"], state["refs"], out_frame_idx)
                    overlay_path = os.path.join(state["workdir"], f"overlay_{out_frame_idx:05d}.jpg")
                    im.save(overlay_path)
                state["overlay_paths"][out_frame_idx] = overlay_path
                try:
                    pil_img = self._pil_from_path(overlay_path)
                except Exception:
                    pil_img = self._pil_from_path(frame_path)
                yield f"Propagated frame {out_frame_idx+1}/{n}", gr.update(value=pil_img), gr.update(value=out_frame_idx)
        except Exception as e:
            state["busy"] = False
            yield f"Propagation failed: {e}", gr.update(value=self._pil_from_path(state["frame_paths"][last_idx])), gr.update(value=last_idx)
            return
        for f in range(n):
            if state["overlay_paths"][f] is None:
                masks_dict = state["masks_per_frame"][f]
                frame_path = state["frame_paths"][f]
                if masks_dict:
                    try:
                        state["overlay_paths"][f] = self._overlay_masks_on_image(state, frame_path, masks_dict, id_to_name, frame_idx=f)
                    except Exception as e:
                        print(f"[track_objects] overlay creation failed for frame {f}: {e}")
                        im = draw_points_on_frame(frame_path, state["objects"], state["refs"], f)
                        out_path = os.path.join(state["workdir"], f"overlay_{f:05d}.jpg")
                        im.save(out_path)
                        state["overlay_paths"][f] = out_path
                else:
                    im = draw_points_on_frame(frame_path, state["objects"], state["refs"], f)
                    out_path = os.path.join(state["workdir"], f"overlay_{f:05d}.jpg")
                    im.save(out_path)
                    state["overlay_paths"][f] = out_path
        state["propagated"] = True
        state["busy"] = False
        final_display = state["overlay_paths"][last_idx] if state["overlay_paths"] and 0 <= last_idx < len(state["overlay_paths"]) else state["frame_paths"][last_idx]
        yield f"Done. Overlays built for all frames. {state['workdir']}", gr.update(value=self._pil_from_path(final_display)), gr.update(value=last_idx)

    def on_export_masks(self, state: Dict[str, Any]):
        try:
            workdir = state.get("workdir") or tempfile.gettempdir()
            frame_paths = state.get("frame_paths", [])
            masks_per_frame = state.get("masks_per_frame", [])
            name_to_id = state.get("obj_name_to_id", {})
            id_to_name = {v: k for k, v in name_to_id.items()}
            video_w_px, video_h_px = 720, 1280
            refs = state.get("refs", {"Ref 1": [], "Ref 2": []})
            ref1_pt = refs.get("Ref 1", [])[-1] if refs.get("Ref 1") else None
            ref2_pt = refs.get("Ref 2", [])[-1] if refs.get("Ref 2") else None
            if not ref1_pt or not ref2_pt:
                msg = "Export failed: Need both Ref 1 and Ref 2 points."
                return state, gr.update(visible=False, value=None), msg
            x1, y1, _ = ref1_pt
            x2, y2, _ = ref2_pt
            # dx = abs(float(x2) - float(x1))
            dy = abs(float(y2) - float(y1))
            real_ref_len_cm = 120.0
            # cm_per_px_x = (real_ref_len_cm / dx) if dx > 0 else None
            cm_per_px_y = (real_ref_len_cm / dy) if dy > 0 else None
            cm_per_px_x = cm_per_px_y
            baseline_name = None
            baseline_id = None
            if "Object 1" in name_to_id:
                baseline_name = "Object 1"
                baseline_id = name_to_id[baseline_name]
            elif name_to_id:
                baseline_id = sorted(name_to_id.values())[0]
                baseline_name = id_to_name.get(baseline_id, f"Object {baseline_id}")
            else:
                export = {
                    "video": {"width_px": video_w_px, "height_px": video_h_px},
                    "reference": {
                        "ref1_screen": [x1, y1],
                        "ref2_screen": [x2, y2],
                        "real_distance_cm": real_ref_len_cm,
                        "scale_cm_per_px": {"x": cm_per_px_x, "y": cm_per_px_y},
                    },
                    "frames": []
                }
                export_path = os.path.join(workdir, f"export_masks_{uuid.uuid4().hex[:8]}.json")
                with open(export_path, "w", encoding="utf-8") as f:
                    json.dump(export, f, ensure_ascii=False, indent=2)
                return state, gr.update(value=export_path, visible=True), f"Exported (meta only): {export_path}"
            def _mask_center(mask: np.ndarray):
                ys, xs = np.where(mask.astype(np.uint8) > 0)
                if xs.size == 0:
                    return None
                cx = float(xs.mean())
                cy = float(ys.mean())
                return [cx, cy]
            frames_out = []
            num_frames = len(frame_paths)
            for fidx in range(num_frames):
                frame_entry = {"frame_index": fidx, "objects": {}}
                masks_dict = masks_per_frame[fidx] if fidx < len(masks_per_frame) else {}
                base_center_scr = None
                base_mask = masks_dict.get(baseline_id)
                if base_mask is not None:
                    base_center_scr = _mask_center(base_mask)
                if base_center_scr is not None:
                    frame_entry["objects"][baseline_name] = {
                        "center_screen": [round(base_center_scr[0], 3), round(base_center_scr[1], 3)],
                        "center_real_cm": [0.0, 0.0]
                    }
                else:
                    frame_entry["objects"][baseline_name] = {
                        "center_screen": None,
                        "center_real_cm": None
                    }
                for obj_id, mask in masks_dict.items():
                    if obj_id == baseline_id:
                        continue
                    obj_name = id_to_name.get(obj_id, f"Object {obj_id}")
                    mask_u8 = (mask.astype(np.uint8) * 255) if mask.dtype != np.uint8 else mask
                    try:
                        contours, _ = cv2.findContours(mask_u8, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
                    except ValueError:
                        _, contours, _ = cv2.findContours(mask_u8, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
                    polygons_screen = []
                    polygons_real = []
                    for cnt in contours:
                        if cnt is None or len(cnt) < 3:
                            continue
                        peri = cv2.arcLength(cnt, True)
                        epsilon = max(1.0, 0.002 * float(peri))
                        approx = cv2.approxPolyDP(cnt, epsilon, True)
                        pts = approx.reshape(-1, 2).astype(float)
                        poly_scr = [[float(round(p[0], 3)), float(round(p[1], 3))] for p in pts]
                        polygons_screen.append(poly_scr)
                        if base_center_scr is not None and cm_per_px_x and cm_per_px_y:
                            cx, cy = base_center_scr
                            poly_real = []
                            for p in pts:
                                x, y = float(p[0]), float(p[1])
                                rx = (x - cx) * cm_per_px_x
                                ry = (cy - y) * cm_per_px_y
                                poly_real.append([float(round(rx, 3)), float(round(ry, 3))])
                            polygons_real.append(poly_real)
                        else:
                            polygons_real.append(None)
                    frame_entry["objects"][obj_name] = {
                        "polygons_screen": polygons_screen,
                        "polygons_real_cm": polygons_real
                    }
                frames_out.append(frame_entry)
            export = {
                "video": {"width_px": video_w_px, "height_px": video_h_px},
                "reference": {
                    "ref1_screen": [float(x1), float(y1)],
                    "ref2_screen": [float(x2), float(y2)],
                    "real_distance_cm": real_ref_len_cm,
                    "scale_cm_per_px": {"x": cm_per_px_x, "y": cm_per_px_y},
                },
                "baseline_object": baseline_name,
                "frames": frames_out
            }
            export_path = os.path.join(workdir, f"export_masks_{uuid.uuid4().hex[:8]}.json")
            with open(export_path, "w", encoding="utf-8") as f:
                json.dump(export, f, ensure_ascii=False, indent=2)
            msg = f"Exported: {export_path} (frames={len(frames_out)}, objects={len(name_to_id)})"
            return state, gr.update(value=export_path, visible=True), msg
        except Exception as e:
            return state, gr.update(visible=False, value=None), f"Export failed: {e}"

    # -----------------------------
    # UI builder
    # -----------------------------
    def build_ui(self):
        with gr.Blocks(theme="soft", css=self.css) as demo:
            gr.HTML(self.css_inject)
            gr.Markdown("## Segment-Anything-like UI (Gradio Skeleton)")
            st = gr.State({
                "workdir": None,
                "frame_paths": [],
                "objects": {},
                "refs": {"Ref 1": [], "Ref 2": []},
                "inference_state": None,
                "obj_name_to_id": {},
                "next_obj_id": 1,
                "obj_colors": {},
                "masks_per_frame": [],
                "overlay_paths": [],
                "propagated": False,
                "busy": False,
            })
            with gr.Row():
                with gr.Column(scale=1, elem_id="left_col"):
                    gr.Markdown("### 🎛️ Toolbar")
                    video = gr.Video(label="Upload a video")
                    process_btn = gr.Button("Process (Extract Frames)")
                    with gr.Accordion("Reference Objects (for coordinates only)", open=True):
                        active_ref = gr.Dropdown(choices=["Ref 1", "Ref 2"], value="Ref 1", label="Active reference")
                        clear_ref_btn = gr.Button("Clear points of active reference")
                    with gr.Accordion("Objects to Segment", open=True):
                        active_object = gr.Dropdown(choices=[], label="Active object", value=None)
                        add_object_btn = gr.Button("Add object")
                        remove_object_btn = gr.Button("Remove active object")
                        point_mode = gr.Radio(choices=["Add point", "Remove point"], value="Add point", label="Point mode")
                        clear_points_btn = gr.Button("Clear points of active object")
                    track_btn = gr.Button("🚀 Track Objects")
                    export_btn = gr.Button("Export Masks (.json)")
                with gr.Column(scale=3, elem_id="right_col"):
                    gr.Markdown("### 📹 Video & Frame Viewer")
                    with gr.Row():
                        frame_img = gr.Image(label="Click on the frame (coordinates captured here only)", interactive=True, type="pil", elem_id="frame_img")
                    frame_slider = gr.Slider(0, 0, value=0, step=1, label="Frame index", interactive=True)
                    click_info = gr.HTML("Click info will appear here.")
                    progress_log = gr.Textbox(label="Progress log", lines=6)
                    export_file = gr.File(label="Exported JSON", visible=False)
            # Wire events
            process_btn.click(self.on_process, inputs=[video, st], outputs=[st, frame_img, frame_slider, active_object, progress_log])
            add_object_btn.click(self.on_add_object, inputs=[st], outputs=[st, active_object])
            remove_object_btn.click(self.on_remove_active_object, inputs=[st, active_object, frame_slider], outputs=[st, active_object, frame_img, progress_log, click_info])
            clear_points_btn.click(self.on_clear_object_points, inputs=[st, active_object, frame_slider], outputs=[st, progress_log, frame_img])
            clear_ref_btn.click(self.on_clear_ref_points, inputs=[st, active_ref, frame_slider], outputs=[st, progress_log, frame_img])
            frame_slider.change(self.on_frame_change, inputs=[st, frame_slider], outputs=[st, frame_img])
            frame_img.select(self.on_click_frame, inputs=[st, active_object, point_mode, active_ref, frame_slider], outputs=[st, frame_img, click_info])
            track_btn.click(self.track_objects, inputs=[st], outputs=[progress_log, frame_img, frame_slider])
            export_btn.click(self.on_export_masks, inputs=[st], outputs=[st, export_file, progress_log])
        return demo


In [ ]:
# -----------------------------
# App Runner (Class-based)
# -----------------------------

# Build the Gradio Blocks UI using the encapsulated DrapeApp
app = DrapeApp(predictor=predictor, css=CSS, css_inject=css_inject)
demo = app.build_ui()

# Launch Gradio app (same behavior as before, now separated from implementation)
demo.launch()